# Alerte Pluie Guinée : notebook de démarrage
**IndabaX Guinée 2026, Hackathon**

Ce notebook donne une **baseline volontairement naïve** : aucun nettoyage, aucune variable temporelle, aucun fichier additionnel.
Son score se situe autour de **0,60 de log loss**. À vous de faire beaucoup mieux.

Pistes : relisez `description.txt`, explorez les données (EDA), et demandez-vous ce qu'un météorologue regarderait pour prévoir la pluie de demain.

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import log_loss

CLASSES = ["aucune", "faible", "moderee", "forte"]
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sub = pd.read_csv("sample_submission.csv")
print(train.shape, test.shape, sub.shape)
train.head()

In [ ]:
train.target.value_counts(normalize=True)

## Variables très simples

In [ ]:
def simple_features(d):
    X = d[["temperature", "heat_index", "comfort_index", "humidity", "pressure",
           "wind_speed", "wind_deg", "cloud_cover", "rain_3h"]].copy()
    t = pd.to_datetime(d.ingestion_timestamp)
    X["hour"] = t.dt.hour
    X["month"] = t.dt.month
    X["main_condition"] = d.main_condition.astype("category")
    return X

X = simple_features(train)
y = train.target.map({c: i for i, c in enumerate(CLASSES)})

## Validation
Attention : le choix de la validation est **crucial** dans ce challenge. Ici on utilise une découpe aléatoire. Le test, lui, porte sur une **autre année** et contient **des villes absentes du train**. Cette validation reflète-t-elle vraiment la situation du test ?

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
params = dict(objective="multiclass", num_class=4, learning_rate=0.05, num_leaves=31, verbose=-1)
model = lgb.train(params, lgb.Dataset(X_tr, y_tr), 300)
print("log loss validation (aléatoire) :", log_loss(y_va, model.predict(X_va)))

## Entraînement complet et soumission

In [ ]:
model = lgb.train(params, lgb.Dataset(X, y), 300)
to_predict = test[test.ID.isin(sub.ID)]
proba = model.predict(simple_features(to_predict))
submission = pd.DataFrame(proba, columns=CLASSES)
submission.insert(0, "ID", to_predict.ID.values)
submission = sub[["ID"]].merge(submission, on="ID", how="left")
assert submission[CLASSES].notna().all().all()
submission.to_csv("ma_premiere_soumission.csv", index=False)
submission.head()